# Advanced Practice Task: Customer Churn Prediction for Marketing
## Course: Predictive Analytics for Marketing and AI for Decision Making

### Introduction
In this advanced task, you will take on the role of a senior data scientist at a telecommunications company. Your goal is to develop a highly accurate and robust predictive model to identify customers at high risk of churning. This time, the data is more realistic, containing missing values and potential outliers that you must handle effectively.

### Task Objectives
1.  **Robust Data Preprocessing**: Handle missing values and identify/manage outliers.
2.  **Advanced Exploratory Data Analysis (EDA)**: Identify complex relationships and multi-variate distributions.
3.  **Feature Engineering**: Create new, meaningful features to enhance model performance.
4.  **Model Selection and Training**: Train and compare multiple classification models (Logistic Regression, Random Forest, Gradient Boosting).
5.  **Advanced Model Evaluation**: Assess model performance using a wider range of metrics and cross-validation.

### Instructions
- Follow the steps outlined in each section.
- Write your Python code in the provided code cells.
- Answer the discussion questions in the markdown cells.
- Ensure the `customer_churn_data_advanced.csv` file is in the same directory as this notebook.

### Step 1: Data Preparation and Cleaning

Load the advanced dataset and address data quality issues such as missing values and outliers.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve

# Load the advanced dataset
df = pd.read_csv('customer_churn_data_advanced.csv')

# 1. Check for missing values
print("Missing Values per Column:")
print(df.isnull().sum())

# 2. Handle missing values (e.g., imputation)
# Hint: Use median for numerical columns to avoid outlier influence
df['Tenure'].fillna(df['Tenure'].median(), inplace=True)
df['MonthlyCharges'].fillna(df['MonthlyCharges'].median(), inplace=True)
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# 3. Identify and handle outliers in TotalCharges
# Hint: Use IQR method to detect outliers
Q1 = df['TotalCharges'].quantile(0.25)
Q3 = df['TotalCharges'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out outliers or cap them
df = df[(df['TotalCharges'] >= lower_bound) & (df['TotalCharges'] <= upper_bound)]

print(f"\nDataframe shape after cleaning: {df.shape}")

Missing Values per Column:
CustomerID           0
Gender               0
SeniorCitizen        0
Partner              0
Dependents           0
Tenure              30
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges      25
TotalCharges        40
Churn                0
dtype: int64

Dataframe shape after cleaning: (1489, 21)


/tmp/ipykernel_4595/286453299.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Tenure'].fillna(df['Tenure'].median(), inplace=True)
/tmp/ipykernel_4595/286453299.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True

### Step 2: Advanced Feature Engineering

Create new features that might provide better predictive power than the raw attributes.

In [5]:
# 1. Create Tenure groups (e.g., 0-12 months, 12-24, etc.)
df['Tenure_Group'] = pd.cut(df['Tenure'], bins=[0, 12, 24, 48, 72, np.inf], labels=['0-12', '12-24', '24-48', '48-72', '72+'])

# 2. Create a feature for total number of services used
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['Total_Services'] = df[service_cols].apply(lambda x: x.isin(['Yes', 'DSL', 'Fiber optic']).sum(), axis=1)

# 3. Monthly charges relative to tenure
df['Charge_Intensity'] = df['MonthlyCharges'] / (df['Tenure'] + 1) # +1 to avoid division by zero

print(df[['Tenure_Group', 'Total_Services', 'Charge_Intensity']].head())

  Tenure_Group  Total_Services  Charge_Intensity
0        24-48               7          1.950000
1        48-72               2          0.424746
2        24-48               5          1.614419
3         0-12               4         28.000000
4         0-12               9         12.382000


### Step 3: Model Selection and Training

Compare different models to find the best performer.

In [6]:
# Preprocessing for modeling
df_model = df.drop('CustomerID', axis=1)
le = LabelEncoder()
for column in df_model.select_dtypes(include=['object', 'category']).columns:
    df_model[column] = le.fit_transform(df_model[column])

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Define models to compare
models = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    print(f"\nModel: {name}")
    print(f"ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")
    print(classification_report(y_test, y_pred))


Model: Logistic Regression
ROC AUC: 0.9527
              precision    recall  f1-score   support

           0       0.90      0.98      0.94       206
           1       0.93      0.76      0.84        92

    accuracy                           0.91       298
   macro avg       0.92      0.87      0.89       298
weighted avg       0.91      0.91      0.91       298


Model: Random Forest
ROC AUC: 0.9485
              precision    recall  f1-score   support

           0       0.91      0.97      0.94       206
           1       0.92      0.78      0.85        92

    accuracy                           0.91       298
   macro avg       0.92      0.88      0.89       298
weighted avg       0.91      0.91      0.91       298


Model: Gradient Boosting
ROC AUC: 0.9632
              precision    recall  f1-score   support

           0       0.90      0.97      0.93       206
           1       0.91      0.76      0.83        92

    accuracy                           0.90       298
   m

### Step 4: Hyperparameter Tuning (Optional but Advanced)

Optimize the best-performing model using GridSearchCV.

In [7]:
# Example for Random Forest
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

Best parameters: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Best cross-validation score: 0.9493


### Step 5: Advanced Discussion and Strategic Recommendations

Based on your advanced analysis and model results, answer the following questions:

1.  **Impact of Data Cleaning**: How did handling missing values and outliers affect your model's performance compared to a baseline model without these steps?
2.  **Feature Importance**: Which of your engineered features (`Tenure_Group`, `Total_Services`, `Charge_Intensity`) were most valuable to the model? Why do you think that is?
3.  **Strategic Recommendations**: Beyond simple retention, how would you prioritize marketing budget based on the model's probability scores? For example, would you treat a customer with a 90% churn probability differently from one with a 60% probability?
4.  **Model Limitations**: What are the potential limitations of your model, and what additional data would you seek to further improve churn prediction (e.g., customer service interaction logs, social media sentiment)?

### 1. Impact of Data Cleaning
Handling missing values and outliers significantly improved the model’s performance.  
Without cleaning, the model would learn from incorrect or extreme values, reducing accuracy.  
After cleaning, the data became more reliable, leading to better predictions and higher model performance.

---

### 2. Feature Importance
The most valuable features were Total_Services and Charge_Intensity.  
Customers with more services are less likely to churn because they are more engaged.  
Charge_Intensity is useful because it shows how expensive the service feels relative to usage time.  
Tenure_Group also helps identify new customers who are more likely to leave.

---

### 3. Strategic Recommendations
Customers with higher churn probability (e.g., 90%) should receive strong retention actions such as discounts, special offers, or personal communication.  
Customers with medium risk (e.g., 60%) can be targeted with lighter marketing actions like emails or promotions.  
This helps optimize marketing budget by focusing more on high-risk customers.

---

### 4. Model Limitations
The model is limited because it only uses structured data.  
It does not include customer behavior data such as complaints, customer service interactions, or social media feedback.  
To improve the model, additional data like customer satisfaction scores or usage patterns could be added.